# Task 2 — User Overview (Part 1)

**Goals for this notebook:**
1. Load xDR session data from PostgreSQL
2. Identify top handsets & manufacturers
3. Aggregate per-user application behaviour (Task 2.1)

> Logic lives in `src/tellco_user_analytics/` — this notebook only orchestrates and visualises.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from tellco_user_analytics.data.loader import load_xdr_sessions
from tellco_user_analytics.analysis.handsets import (
    top_handsets,
    top_manufacturers,
    top_handsets_by_manufacturer,
    marketing_handset_summary,
)
from tellco_user_analytics.analysis.user_overview import aggregate_user_sessions

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

## 1. Load data from PostgreSQL

Each row = one **xDR session** (a data usage event). Customer ID = `MSISDN/Number`.

In [ ]:
df = load_xdr_sessions()
print(f"Sessions loaded: {len(df):,}")
print(f"Unique customers: {df['MSISDN/Number'].nunique():,}")
df.head(3)

## 2. Handset analysis

Identify the most popular devices and manufacturers to guide marketing.

In [ ]:
top_10_handsets = top_handsets(df, n=10)
top_3_manufacturers = top_manufacturers(df, n=3)
handsets_by_mfr = top_handsets_by_manufacturer(df, top_3_manufacturers.index, n=5)

display(top_10_handsets.to_frame("session_count"))
display(top_3_manufacturers.to_frame("session_count"))
for mfr, models in handsets_by_mfr.items():
    print(f"\n{mfr} — top 5 models:")
    display(models.to_frame("session_count"))

print("\n" + marketing_handset_summary(df))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_10_handsets.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Top 10 Handset Models")
axes[0].set_xlabel("Session count")
axes[0].invert_yaxis()

top_3_manufacturers.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Top 3 Manufacturers")
axes[1].set_ylabel("Session count")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

## 3. Task 2.1 — Per-user application behaviour

Aggregate sessions to one row per customer with total traffic per app.

In [ ]:
user_agg = aggregate_user_sessions(df)
print(f"Aggregated users: {len(user_agg):,}")
user_agg.head()